# Convert PyKeen Datasets to External Formats
Utility methods that convert a PyKeen knowledge graph to a PyG Data object or PyTorch Tensors. Three files (train, val, test) are saved for each PyKeen knowledge graph. There are also utility methods to save the entity index to name and relation index to name mappings.

In [16]:
from torch_geometric.data import Data
import torch
from pykeen.datasets import WN18RR, Countries
from os import path

Methods to save the entity / relation index to string name mappings.

In [ ]:
def save_entity_name_mapping(factory, filename):
    with open(filename, "w") as f:
        first_line = True
        for k,v in factory.entity_to_id.items():
            if first_line:
                f.write(f"{v}\t{k}")
                first_line = False
            else:
                f.write(f"\n{v}\t{k}")

def save_relation_name_mapping(factory, filename):
    with open(filename, "w") as f:
        first_line = True
        for k,v in factory.relation_to_id.items():
            if first_line:
                f.write(f"{v}\t{k}")
                first_line = False
            else:
                f.write(f"\n{v}\t{k}")

# if the dataset is transductive, the splits should all have the same mappings
def save_all_entity_relation_name_mappings(dataset, file_namer):
    factory = dataset.training
    save_entity_name_mapping(factory, file_namer("entity", "train"))
    save_relation_name_mapping(factory, file_namer("relation", "train"))
    factory = dataset.validation
    save_entity_name_mapping(factory, file_namer("entity", "val"))
    save_relation_name_mapping(factory, file_namer("relation", "val"))
    factory = dataset.testing
    save_entity_name_mapping(factory, file_namer("entity", "test"))
    save_relation_name_mapping(factory, file_namer("relation", "test"))

def make_entity_relation_namer(directory, dataset_name):
    def namer(etype, split):
        return path.join(directory, f"{dataset_name}_{etype}_{split}.tsv")
    return namer

Methods to save the triples tensors for train, val, test to .py files.

In [46]:
def save_triples_tensor(factory, filename):
    triples = factory.mapped_triples
    torch.save(triples, filename)

def save_all_triples_tensors(dataset, file_namer):
    factory = dataset.training
    save_triples_tensor(factory, file_namer("train"))
    factory = dataset.validation
    save_triples_tensor(factory, file_namer("val"))
    factory = dataset.testing
    save_triples_tensor(factory, file_namer("test"))

def make_triples_namer(directory, dataset_name):
    def namer(split):
        return path.join(directory, f"{dataset_name}_triples_{split}.pt")
    return namer

Methods to convert the triples to a PyTorch Geometric Data object and save to a .py file.

In [42]:
def convert_kg_to_pyg(factory):
    triples_tensor = factory.mapped_triples.long()
    # triples in pykeen are in (head, relation, tail) format
    # pyg expects edge_index in (source, target) format
    # so we need to convert (h, r, t) to (h, t)
    # and store r as edge_type
    edge_index = edge_index =triples_tensor[:, [0, 2]].transpose(0, 1)
    edge_type = triples_tensor[:, 1]
    num_nodes = factory.num_entities
    data = Data(edge_index=edge_index, edge_type=edge_type, num_nodes=num_nodes)
    return data

def save_kg_as_pyg(factory, filename):
    pyg_data = convert_kg_to_pyg(factory)
    torch.save(pyg_data, filename)
    return pyg_data

def convert_and_save_kg(dataset, filename_namer):
    save_kg_as_pyg(factory=dataset.training, filename=filename_namer("train"))
    save_kg_as_pyg(factory=dataset.validation, filename=filename_namer("val"))
    save_kg_as_pyg(factory=dataset.testing, filename=filename_namer("test"))
    return

def make_kg_pyg_namer(directory, dataset_name):
    def namer(split):
        return path.join(directory, f"{dataset_name}_kg_pyg_{split}.pt")
    return namer

In [47]:

fn1 = make_kg_pyg_namer("./data", "countries")
ds = Countries()
convert_and_save_kg(ds, fn1)
fn2 = make_entity_relation_namer("./data", "countries")
save_all_entity_relation_name_mappings(ds, fn2)
fn3 = make_triples_namer("./data", "countries")
save_all_triples_tensors(ds, fn3)

In [34]:
# load the PyG Data object for testing
pyg_data = torch.load(fn1("train"), weights_only=False)
pyg_data

Data(edge_index=[2, 1110], edge_type=[1110], num_nodes=271)

In [48]:
# load the triples tensor for testing
triples_tensor = torch.load(fn3("train"))
triples_tensor

tensor([[  0,   0,  13],
        [  0,   0, 223],
        [  0,   1,  50],
        ...,
        [269,   1, 268],
        [270,   0,  77],
        [270,   0, 173]])